In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP20 - INITIALIZATION
# Normalization / HVG / PCA / Integration Decision
# ============================================================

from pathlib import Path
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import gc
import sys

SEED = 42
np.random.seed(SEED)

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step20_Normalization_Integration')
)

DATA_DIR = ROOT / "Data"
NOTEBOOK_DIR = ROOT / "Notebook"
RESULTS_DIR = ROOT / "Results"
FIGURES_DIR = ROOT / "Figures"
LOGS_DIR = ROOT / "Logs"

for p in [
    DATA_DIR,
    NOTEBOOK_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

INPUT_H5AD = Path(
    _hcc_path('LIHC_scRNA_Step19_QC/Results/GSE149614_CLEAN_COUNTS.h5ad')
)

assert INPUT_H5AD.exists()

print("=" * 90)
print("STEP20 INITIALIZED")
print("=" * 90)

print("Python:")
print(sys.executable)

print("\nScanpy:")
print(sc.__version__)

print("\nInput:")
print(INPUT_H5AD)

print("\nExists:")
print(INPUT_H5AD.exists())

In [ ]:
# ============================================================
# STEP20 - LOAD CLEAN RAW COUNTS
# ============================================================

adata = sc.read_h5ad(
    INPUT_H5AD
)

print("=" * 90)
print("STEP20 INPUT AUDIT")
print("=" * 90)

print(adata)

print("\nShape:")
print(adata.shape)

print("\nCells:")
print(f"{adata.n_obs:,}")

print("\nGenes:")
print(f"{adata.n_vars:,}")

print("\nSamples:")
print(adata.obs["sample"].nunique())

print("\nPatients:")
print(adata.obs["patient"].nunique())

print("\nSites:")
print(
    adata.obs["site"]
    .value_counts()
)

In [ ]:
# ============================================================
# STEP20 - RAW COUNT INTEGRITY CHECK
# ============================================================

if hasattr(
    adata.X,
    "data"
):

    values = (
        adata.X[:1000, :1000]
        .data
    )

else:

    values = np.asarray(
        adata.X[:1000, :1000]
    ).ravel()


if len(values) > 0:

    integer_fraction = np.mean(
        np.isclose(
            values,
            np.round(values)
        )
    )

else:

    integer_fraction = 1.0


print("=" * 90)
print("RAW COUNT INTEGRITY")
print("=" * 90)

print(
    "Integer fraction:",
    f"{integer_fraction:.4f}"
)

print(
    "Minimum:",
    adata.X.min()
)

print(
    "Maximum:",
    adata.X.max()
)

assert integer_fraction > 0.99
assert adata.X.min() >= 0

print("\nRAW COUNTS = PASS")

In [ ]:
# ============================================================
# STEP20 - RECORD RAW COUNT PROVENANCE
# ============================================================

adata.uns[
    "raw_counts_source"
] = str(
    INPUT_H5AD
)

adata.uns[
    "raw_counts_preserved_externally"
] = True

adata.uns[
    "step20_note"
] = (
    "Raw integer counts are preserved in the "
    "Step19 CLEAN_COUNTS checkpoint. "
    "Step20 X will be normalized/log-transformed."
)

print(
    adata.uns[
        "raw_counts_source"
    ]
)

In [ ]:
# ============================================================
# STEP20 - LIBRARY-SIZE NORMALIZATION
# ============================================================

print("=" * 90)
print("NORMALIZATION")
print("=" * 90)

print(
    "Before normalization:"
)

before_totals = np.asarray(
    adata.X.sum(axis=1)
).ravel()

print(
    "Median total counts:",
    np.median(
        before_totals
    )
)

print(
    "Min:",
    before_totals.min()
)

print(
    "Max:",
    before_totals.max()
)


# Normalize each cell to total = 10,000
sc.pp.normalize_total(
    adata,
    target_sum=1e4
)


after_totals = np.asarray(
    adata.X.sum(axis=1)
).ravel()

print(
    "\nAfter normalization:"
)

print(
    "Median total:",
    np.median(
        after_totals
    )
)

print(
    "Min:",
    after_totals.min()
)

print(
    "Max:",
    after_totals.max()
)

print("\nNORMALIZATION = COMPLETE")

In [ ]:
# ============================================================
# STEP20 - LOG1P TRANSFORMATION
# ============================================================

sc.pp.log1p(
    adata
)

adata.uns[
    "normalization"
] = {
    "method":
        "library_size_normalization_plus_log1p",

    "target_sum":
        10000,

    "log_transform":
        "log1p"
}

print("=" * 90)
print("LOG1P TRANSFORMATION COMPLETE")
print("=" * 90)

print(
    "X minimum:",
    adata.X.min()
)

print(
    "X maximum:",
    adata.X.max()
)

In [ ]:
# ============================================================
# STEP20 - SAVE LOG-NORMALIZED CHECKPOINT
# ============================================================

LOGNORM_H5AD = (
    DATA_DIR /
    "GSE149614_LOGNORM_ALL_GENES.h5ad"
)

adata.write_h5ad(
    LOGNORM_H5AD,
    compression="gzip"
)

print("=" * 90)
print("LOG-NORMALIZED CHECKPOINT SAVED")
print("=" * 90)

print(LOGNORM_H5AD)

print(
    "\nShape:",
    adata.shape
)

print(
    "File size:",
    f"{LOGNORM_H5AD.stat().st_size / 1024**3:.3f} GB"
)

In [ ]:
# ============================================================
# STEP20 - HIGHLY VARIABLE GENES
# Batch-aware by sample
# ============================================================

print("=" * 90)
print("HIGHLY VARIABLE GENE SELECTION")
print("=" * 90)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat",
    batch_key="sample",
    inplace=True
)

n_hvg = int(
    adata.var[
        "highly_variable"
    ].sum()
)

print(
    "Selected HVGs:",
    f"{n_hvg:,}"
)

print(
    "Total genes:",
    f"{adata.n_vars:,}"
)

assert n_hvg > 0

print("\nHVG = COMPLETE")

In [ ]:
# ============================================================
# STEP20 - SAVE HVG TABLE
# ============================================================

hvg_cols = [
    c for c in [
        "highly_variable",
        "means",
        "dispersions",
        "dispersions_norm",
        "highly_variable_nbatches"
    ]
    if c in adata.var.columns
]

hvg_table = (
    adata.var[
        hvg_cols
    ]
    .copy()
)

hvg_table.index.name = "Gene"

hvg_table.to_csv(
    RESULTS_DIR /
    "Step20_HVG_AllGenes.csv",
    encoding="utf-8-sig"
)

hvg_only = (
    hvg_table[
        hvg_table[
            "highly_variable"
        ]
    ]
    .copy()
)

hvg_only.to_csv(
    RESULTS_DIR /
    "Step20_HVG_Selected3000.csv",
    encoding="utf-8-sig"
)

print(
    "HVG table saved."
)

print(
    "Selected:",
    len(hvg_only)
)

In [ ]:
# ============================================================
# STEP20 - BULK ANCHOR × HVG AUDIT
# ============================================================

ORIGINAL_TOP10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"
]

HUB18 = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]


rows = []

for set_name, genes in [
    ("OriginalTop10", ORIGINAL_TOP10),
    ("WGCNA_Hub18", HUB18)
]:

    for gene in genes:

        present = (
            gene in adata.var_names
        )

        if present:

            is_hvg = bool(
                adata.var.loc[
                    gene,
                    "highly_variable"
                ]
            )

        else:

            is_hvg = False

        rows.append({
            "FeatureSet":
                set_name,

            "Gene":
                gene,

            "Detected":
                present,

            "HighlyVariable":
                is_hvg
        })


anchor_hvg = pd.DataFrame(
    rows
)

display(
    anchor_hvg
)

anchor_hvg.to_csv(
    RESULTS_DIR /
    "Step20_BulkAnchor_HVG_Audit.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# GLOBAL PUBLICATION FIGURE STYLE
# Nature / NPG-inspired palette
# Apply to Step20 and subsequent analyses
# ============================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc
from pathlib import Path


# ------------------------------------------------------------
# 1. Nature / NPG-inspired colors
# ------------------------------------------------------------

NATURE = {
    "red":      "#E64B35",
    "cyan":     "#4DBBD5",
    "green":    "#00A087",
    "navy":     "#3C5488",
    "salmon":   "#F39B7F",
    "purple":   "#8491B4",
    "mint":     "#91D1C2",
    "darkred":  "#DC0000",
    "brown":    "#7E6148",
    "beige":    "#B09C85",

    "grey":     "#BDBDBD",
    "darkgrey": "#4D4D4D",
    "lightgrey":"#E6E6E6",
}


NATURE10 = [
    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#DC0000",
    "#7E6148",
    "#B09C85",
]


# ------------------------------------------------------------
# 2. Fixed biological colors
# ------------------------------------------------------------

SITE_COLORS = {
    "Normal": "#4DBBD5",
    "Tumor":  "#E64B35",
    "PVTT":   "#00A087",
    "Lymph":  "#3C5488",
}


CELLTYPE_COLORS = {
    "Hepatocyte":  "#E64B35",
    "T/NK":        "#4DBBD5",
    "Myeloid":     "#00A087",
    "B":           "#3C5488",
    "Endothelial": "#F39B7F",
    "Fibroblast":  "#8491B4",
}


# ------------------------------------------------------------
# 3. Matplotlib global style
# ------------------------------------------------------------

mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.family": "Arial",
    "font.size": 9,

    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,

    "legend.fontsize": 8,

    "axes.linewidth": 0.8,

    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,

    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.grid": False,
})


# ------------------------------------------------------------
# 4. Scanpy settings
# ------------------------------------------------------------

sc.settings.set_figure_params(
    dpi=120,
    dpi_save=600,
    fontsize=9,
    facecolor="white",
    transparent=False
)


# ------------------------------------------------------------
# 5. Unified figure-saving function
# ------------------------------------------------------------

def save_publication_figure(
    fig,
    filename,
    figure_dir=FIGURES_DIR,
    dpi=600
):
    """
    Save both high-resolution PNG and vector PDF.
    filename should NOT contain extension.
    """

    figure_dir = Path(figure_dir)

    png_path = (
        figure_dir /
        f"{filename}.png"
    )

    pdf_path = (
        figure_dir /
        f"{filename}.pdf"
    )

    # IMPORTANT:
    # save before plt.show()
    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )

    print("Saved:")
    print(png_path)
    print(pdf_path)


print("=" * 80)
print("PUBLICATION FIGURE STYLE LOADED")
print("=" * 80)
print("PNG dpi = 600")
print("PDF vector output = enabled")
print("Palette = Nature/NPG-inspired")

In [ ]:
# ============================================================
# STEP20 - HVG FIGURE
# Publication-quality Nature/NPG-inspired style
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

hvg_plot_df = (
    adata.var[
        [
            "means",
            "dispersions_norm",
            "highly_variable"
        ]
    ]
    .copy()
)

hvg_plot_df = (
    hvg_plot_df[
        np.isfinite(
            hvg_plot_df["means"]
        )
        &
        np.isfinite(
            hvg_plot_df["dispersions_norm"]
        )
    ]
)

# log x-axis cannot contain <=0
hvg_plot_df = (
    hvg_plot_df[
        hvg_plot_df["means"] > 0
    ]
)


non_hvg = (
    hvg_plot_df[
        ~hvg_plot_df[
            "highly_variable"
        ]
    ]
)

hvg = (
    hvg_plot_df[
        hvg_plot_df[
            "highly_variable"
        ]
    ]
)


print(
    "Non-HVG:",
    f"{len(non_hvg):,}"
)

print(
    "HVG:",
    f"{len(hvg):,}"
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(6.2, 5.2)
)


# Non-HVG background
ax.scatter(
    non_hvg["means"],
    non_hvg["dispersions_norm"],

    s=7,

    c=NATURE["grey"],

    alpha=0.35,

    edgecolors="none",

    rasterized=True,

    label=f"Other genes (n={len(non_hvg):,})"
)


# HVGs foreground
ax.scatter(
    hvg["means"],
    hvg["dispersions_norm"],

    s=10,

    c=NATURE["red"],

    alpha=0.80,

    edgecolors="none",

    rasterized=True,

    label=f"HVGs (n={len(hvg):,})"
)


# ------------------------------------------------------------
# Axes
# ------------------------------------------------------------

ax.set_xscale("log")

ax.set_xlabel(
    "Mean expression"
)

ax.set_ylabel(
    "Normalized dispersion"
)

ax.set_title(
    "Highly variable gene selection",
    pad=10
)


# Nature-like minimal frame
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

ax.legend(
    frameon=False,
    loc="upper right",
    markerscale=1.5
)


# ------------------------------------------------------------
# Annotation
# ------------------------------------------------------------

ax.text(
    0.02,
    0.98,

    f"{len(hvg):,} HVGs selected",

    transform=ax.transAxes,

    ha="left",
    va="top",

    fontsize=9,

    color=NATURE["darkgrey"]
)


fig.tight_layout()


# ------------------------------------------------------------
# SAVE FIRST
# ------------------------------------------------------------

save_publication_figure(
    fig,
    "Step20_01_HVG_Selection"
)


# ------------------------------------------------------------
# SHOW AFTER SAVING
# ------------------------------------------------------------

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP20 - HVG FIGURE 1
# Mean expression vs raw dispersion
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

plot_df = adata.var[
    [
        "means",
        "dispersions",
        "highly_variable"
    ]
].copy()

plot_df = plot_df[
    np.isfinite(plot_df["means"])
    &
    np.isfinite(plot_df["dispersions"])
    &
    (plot_df["means"] > 0)
]

non_hvg = plot_df[
    ~plot_df["highly_variable"]
]

hvg = plot_df[
    plot_df["highly_variable"]
]


fig, ax = plt.subplots(
    figsize=(6.2, 5.2)
)

# Non-HVG
ax.scatter(
    non_hvg["means"],
    non_hvg["dispersions"],
    s=7,
    c=NATURE["grey"],
    alpha=0.35,
    edgecolors="none",
    rasterized=True,
    label=f"Other genes (n={len(non_hvg):,})"
)

# HVG
ax.scatter(
    hvg["means"],
    hvg["dispersions"],
    s=10,
    c=NATURE["red"],
    alpha=0.80,
    edgecolors="none",
    rasterized=True,
    label=f"HVGs (n={len(hvg):,})"
)

ax.set_xscale("log")

ax.set_xlabel(
    "Mean expression"
)

ax.set_ylabel(
    "Raw dispersion"
)

ax.set_title(
    "Gene dispersion before normalization",
    pad=10
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    loc="best",
    markerscale=1.5
)

fig.tight_layout()

save_publication_figure(
    fig,
    "Step20_01A_HVG_RawDispersion"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP20 - HVG FIGURE 2
# Mean expression vs normalized dispersion
# ============================================================

plot_df = adata.var[
    [
        "means",
        "dispersions_norm",
        "highly_variable"
    ]
].copy()

plot_df = plot_df[
    np.isfinite(plot_df["means"])
    &
    np.isfinite(plot_df["dispersions_norm"])
    &
    (plot_df["means"] > 0)
]

non_hvg = plot_df[
    ~plot_df["highly_variable"]
]

hvg = plot_df[
    plot_df["highly_variable"]
]


fig, ax = plt.subplots(
    figsize=(6.2, 5.2)
)

# Non-HVG
ax.scatter(
    non_hvg["means"],
    non_hvg["dispersions_norm"],
    s=7,
    c=NATURE["grey"],
    alpha=0.35,
    edgecolors="none",
    rasterized=True,
    label=f"Other genes (n={len(non_hvg):,})"
)

# HVG
ax.scatter(
    hvg["means"],
    hvg["dispersions_norm"],
    s=10,
    c=NATURE["red"],
    alpha=0.80,
    edgecolors="none",
    rasterized=True,
    label=f"HVGs (n={len(hvg):,})"
)

ax.set_xscale("log")

ax.set_xlabel(
    "Mean expression"
)

ax.set_ylabel(
    "Normalized dispersion"
)

ax.set_title(
    "Highly variable gene selection",
    pad=10
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    frameon=False,
    loc="best",
    markerscale=1.5
)

ax.text(
    0.02,
    0.98,
    f"{len(hvg):,} HVGs selected",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    color=NATURE["darkgrey"]
)

fig.tight_layout()

save_publication_figure(
    fig,
    "Step20_01B_HVG_NormalizedDispersion"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP20 - CREATE HVG WORKING OBJECT
# ============================================================

adata_hvg = (
    adata[
        :,
        adata.var[
            "highly_variable"
        ]
    ]
    .copy()
)

print(
    adata_hvg
)

print(
    "HVG working shape:",
    adata_hvg.shape
)

In [ ]:
# ============================================================
# STEP20 - SCALE HVGs
# ============================================================

sc.pp.scale(
    adata_hvg,
    max_value=10
)

print("HVG scaling = COMPLETE")

In [ ]:
# ============================================================
# STEP20 - PCA
# ============================================================

sc.tl.pca(
    adata_hvg,
    n_comps=50,
    svd_solver="arpack",
    random_state=42
)

print("=" * 90)
print("PCA COMPLETE")
print("=" * 90)

print(
    "PCA shape:",
    adata_hvg.obsm[
        "X_pca"
    ].shape
)

In [ ]:
# ============================================================
# STEP20 - TRANSFER PCA TO FULL OBJECT
# ============================================================

adata.obsm[
    "X_pca_unintegrated"
] = (
    adata_hvg.obsm[
        "X_pca"
    ]
    .copy()
)

pca_variance = pd.DataFrame({
    "PC":
        np.arange(
            1,
            len(
                adata_hvg.uns[
                    "pca"
                ][
                    "variance_ratio"
                ]
            ) + 1
        ),

    "VarianceRatio":
        adata_hvg.uns[
            "pca"
        ][
            "variance_ratio"
        ]
})

pca_variance[
    "CumulativeVariance"
] = (
    pca_variance[
        "VarianceRatio"
    ]
    .cumsum()
)

display(
    pca_variance.head(20)
)

pca_variance.to_csv(
    RESULTS_DIR /
    "Step20_PCA_Variance.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP20 - PCA VARIANCE FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5)
)

ax.plot(
    pca_variance["PC"],
    pca_variance[
        "CumulativeVariance"
    ],
    marker="o",
    markersize=3
)

ax.set_xlabel(
    "Principal component"
)

ax.set_ylabel(
    "Cumulative explained variance"
)

ax.set_title(
    "PCA cumulative explained variance"
)

fig.tight_layout()

fig.savefig(
    FIGURES_DIR /
    "Step20_02_PCA_CumulativeVariance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP20 - UNINTEGRATED NEIGHBOR GRAPH
# ============================================================

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30,
    use_rep="X_pca_unintegrated",
    key_added="neighbors_unintegrated",
    random_state=42
)

print(
    "Unintegrated neighbor graph = COMPLETE"
)

In [ ]:
# ============================================================
# STEP20 - UNINTEGRATED UMAP
# ============================================================

sc.tl.umap(
    adata,
    neighbors_key="neighbors_unintegrated",
    random_state=42
)

adata.obsm[
    "X_umap_unintegrated"
] = (
    adata.obsm[
        "X_umap"
    ]
    .copy()
)

print(
    "Unintegrated UMAP shape:",
    adata.obsm[
        "X_umap_unintegrated"
    ].shape
)

In [ ]:
# ============================================================
# STEP20 - SAVE UNINTEGRATED UMAP FIGURES
# ============================================================

plot_items = [
    (
        "sample",
        "Step20_03_UMAP_Unintegrated_BySample.png"
    ),
    (
        "patient",
        "Step20_04_UMAP_Unintegrated_ByPatient.png"
    ),
    (
        "site",
        "Step20_05_UMAP_Unintegrated_BySite.png"
    ),
    (
        "celltype",
        "Step20_06_UMAP_Unintegrated_ByOriginalCellType.png"
    )
]

for color_col, filename in plot_items:

    ax = sc.pl.embedding(
        adata,
        basis="umap_unintegrated",
        color=color_col,
        size=5,
        frameon=False,
        show=False
    )

    fig = ax.figure

    fig.savefig(
        FIGURES_DIR /
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

print(
    "Unintegrated UMAP figures saved."
)

In [ ]:
# ============================================================
# STEP20 - DESIGN / CONFOUNDING AUDIT
# ============================================================

sample_site = pd.crosstab(
    adata.obs["sample"],
    adata.obs["site"]
)

patient_site = pd.crosstab(
    adata.obs["patient"],
    adata.obs["site"]
)

print("=" * 90)
print("SAMPLE × SITE")
print("=" * 90)

display(
    sample_site
)

print("\n" + "=" * 90)
print("PATIENT × SITE")
print("=" * 90)

display(
    patient_site
)

sample_site.to_csv(
    RESULTS_DIR /
    "Step20_Sample_Site_Crosstab.csv",
    encoding="utf-8-sig"
)

patient_site.to_csv(
    RESULTS_DIR /
    "Step20_Patient_Site_Crosstab.csv",
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP20 - UNINTEGRATED SILHOUETTE AUDIT
# ============================================================

from sklearn.metrics import silhouette_score

X_pca = (
    adata.obsm[
        "X_pca_unintegrated"
    ][:, :30]
)

rng = np.random.default_rng(42)

n_eval = min(
    10000,
    adata.n_obs
)

eval_idx = rng.choice(
    adata.n_obs,
    size=n_eval,
    replace=False
)

X_eval = X_pca[
    eval_idx
]

metrics = []

for label_col in [
    "sample",
    "patient",
    "site",
    "celltype"
]:

    labels = (
        adata.obs[
            label_col
        ]
        .astype(str)
        .values[
            eval_idx
        ]
    )

    score = silhouette_score(
        X_eval,
        labels,
        metric="euclidean"
    )

    metrics.append({
        "Grouping":
            label_col,

        "Silhouette":
            score,

        "N_cells":
            n_eval
    })


silhouette_summary = pd.DataFrame(
    metrics
)

display(
    silhouette_summary
)

silhouette_summary.to_csv(
    RESULTS_DIR /
    "Step20_Unintegrated_Silhouette_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP20 - WITHIN-CELLTYPE SAMPLE EFFECT
# ============================================================

within_celltype_rows = []

for ct in (
    adata.obs[
        "celltype"
    ]
    .dropna()
    .unique()
):

    mask = (
        adata.obs[
            "celltype"
        ].values == ct
    )

    idx = np.where(
        mask
    )[0]

    sample_labels = (
        adata.obs[
            "sample"
        ]
        .astype(str)
        .values[
            idx
        ]
    )

    n_samples = len(
        np.unique(
            sample_labels
        )
    )

    if (
        len(idx) < 500
        or
        n_samples < 2
    ):
        continue

    if len(idx) > 5000:

        idx_use = rng.choice(
            idx,
            size=5000,
            replace=False
        )

    else:

        idx_use = idx


    labels_use = (
        adata.obs[
            "sample"
        ]
        .astype(str)
        .values[
            idx_use
        ]
    )

    score = silhouette_score(
        X_pca[
            idx_use
        ],
        labels_use,
        metric="euclidean"
    )

    within_celltype_rows.append({
        "CellType":
            ct,

        "N_cells_used":
            len(idx_use),

        "N_samples":
            len(
                np.unique(
                    labels_use
                )
            ),

        "Sample_Silhouette":
            score
    })


within_celltype_asw = pd.DataFrame(
    within_celltype_rows
)

within_celltype_asw = (
    within_celltype_asw
    .sort_values(
        "Sample_Silhouette",
        ascending=False
    )
)

display(
    within_celltype_asw
)

within_celltype_asw.to_csv(
    RESULTS_DIR /
    "Step20_WithinCellType_Sample_Silhouette.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# ============================================================
# STEP20 - SAVE PRE-INTEGRATION CHECKPOINT
# ============================================================

PRE_INTEGRATION_H5AD = (
    DATA_DIR /
    "GSE149614_LOGNORM_HVG_PCA_UMAP_UNINTEGRATED.h5ad"
)

adata.write_h5ad(
    PRE_INTEGRATION_H5AD,
    compression="gzip"
)

print("=" * 90)
print("PRE-INTEGRATION CHECKPOINT SAVED")
print("=" * 90)

print(
    PRE_INTEGRATION_H5AD
)

print(
    "\nShape:",
    adata.shape
)

In [ ]:
# ============================================================
# STEP20 - INTEGRATION DECISION SUMMARY
# ============================================================

print("=" * 100)
print("STEP20 INTEGRATION DECISION SUMMARY")
print("=" * 100)

print("\n[1] DATA")
print(
    f"Cells: {adata.n_obs:,}"
)
print(
    f"Genes: {adata.n_vars:,}"
)

print("\n[2] HVG")
print(
    f"Selected HVGs: "
    f"{int(adata.var['highly_variable'].sum()):,}"
)

print("\n[3] GLOBAL SILHOUETTE")
display(
    silhouette_summary.round(4)
)

print("\n[4] WITHIN-CELLTYPE SAMPLE SILHOUETTE")
display(
    within_celltype_asw.round(4)
)

print("\n[5] SAMPLE × SITE")
display(
    sample_site
)

print("\n[6] PATIENT × SITE")
display(
    patient_site
)

print("\n" + "=" * 100)
print(
    "NORMALIZATION / HVG / PCA / "
    "UNINTEGRATED UMAP = COMPLETE"
)
print(
    "INTEGRATION = NOT YET PERFORMED"
)
print("=" * 100)

In [ ]:
# ============================================================
# STEP20 - OUTPUT SUMMARY FOR INTEGRATION DECISION
# ============================================================

print("=" * 110)
print("STEP20 INTEGRATION DECISION REVIEW")
print("=" * 110)


# ============================================================
# 1. Basic data status
# ============================================================

print("\n[1] DATA STATUS")
print("-" * 110)

print(
    "Cells:",
    f"{adata.n_obs:,}"
)

print(
    "Genes:",
    f"{adata.n_vars:,}"
)

print(
    "Samples:",
    adata.obs["sample"].nunique()
)

print(
    "Patients:",
    adata.obs["patient"].nunique()
)

print(
    "Sites:",
    adata.obs["site"].nunique()
)


# ============================================================
# 2. HVG result
# ============================================================

print("\n[2] HVG SUMMARY")
print("-" * 110)

n_hvg = int(
    adata.var[
        "highly_variable"
    ].sum()
)

print(
    "Selected HVGs:",
    f"{n_hvg:,}"
)

print(
    "Total genes:",
    f"{adata.n_vars:,}"
)


# ============================================================
# 3. PCA result
# ============================================================

print("\n[3] PCA SUMMARY")
print("-" * 110)

print(
    "PCA representation shape:",
    adata.obsm[
        "X_pca_unintegrated"
    ].shape
)

print("\nFirst 20 PCs:")
display(
    pca_variance
    .head(20)
    .round(5)
)


# ============================================================
# 4. Global silhouette
# ============================================================

print("\n[4] GLOBAL SILHOUETTE")
print("-" * 110)

display(
    silhouette_summary
    .round(4)
)


# ============================================================
# 5. Within-celltype sample silhouette
# ============================================================

print("\n[5] WITHIN-CELLTYPE SAMPLE SILHOUETTE")
print("-" * 110)

display(
    within_celltype_asw
    .round(4)
)


# ============================================================
# 6. Sample × Site
# ============================================================

print("\n[6] SAMPLE × SITE")
print("-" * 110)

display(
    sample_site
)


# ============================================================
# 7. Patient × Site
# ============================================================

print("\n[7] PATIENT × SITE")
print("-" * 110)

display(
    patient_site
)


# ============================================================
# 8. UMAP status
# ============================================================

print("\n[8] UNINTEGRATED UMAP")
print("-" * 110)

print(
    "UMAP shape:",
    adata.obsm[
        "X_umap_unintegrated"
    ].shape
)


# ============================================================
# 9. Check important output files
# ============================================================

print("\n[9] SAVED OUTPUT CHECK")
print("-" * 110)

files_to_check = [

    RESULTS_DIR /
    "Step20_HVG_AllGenes.csv",

    RESULTS_DIR /
    "Step20_HVG_Selected3000.csv",

    RESULTS_DIR /
    "Step20_BulkAnchor_HVG_Audit.csv",

    RESULTS_DIR /
    "Step20_PCA_Variance.csv",

    RESULTS_DIR /
    "Step20_Sample_Site_Crosstab.csv",

    RESULTS_DIR /
    "Step20_Patient_Site_Crosstab.csv",

    RESULTS_DIR /
    "Step20_Unintegrated_Silhouette_Summary.csv",

    RESULTS_DIR /
    "Step20_WithinCellType_Sample_Silhouette.csv",

    DATA_DIR /
    "GSE149614_LOGNORM_ALL_GENES.h5ad",

    DATA_DIR /
    "GSE149614_LOGNORM_HVG_PCA_UMAP_UNINTEGRATED.h5ad",
]


for path in files_to_check:

    print(
        "[OK]" if path.exists()
        else "[MISSING]",
        path.name
    )


# ============================================================
# 10. Figure check
# ============================================================

print("\n[10] FIGURE CHECK")
print("-" * 110)

figure_names = [
    "Step20_01A_HVG_RawDispersion",
    "Step20_01B_HVG_NormalizedDispersion",
    "Step20_02_PCA_CumulativeVariance",
    "Step20_03_UMAP_Unintegrated_BySample",
    "Step20_04_UMAP_Unintegrated_ByPatient",
    "Step20_05_UMAP_Unintegrated_BySite",
    "Step20_06_UMAP_Unintegrated_ByOriginalCellType",
]

for name in figure_names:

    png_path = (
        FIGURES_DIR /
        f"{name}.png"
    )

    pdf_path = (
        FIGURES_DIR /
        f"{name}.pdf"
    )

    print(
        name,
        "| PNG:",
        "OK" if png_path.exists() else "MISSING",
        "| PDF:",
        "OK" if pdf_path.exists() else "MISSING"
    )


print("\n" + "=" * 110)
print("STEP20 REVIEW OUTPUT COMPLETE")
print("=" * 110)

print(
    "\nSTATUS:"
)

print(
    "Normalization = COMPLETE"
)

print(
    "HVG selection = COMPLETE"
)

print(
    "PCA = COMPLETE"
)

print(
    "Unintegrated UMAP = COMPLETE"
)

print(
    "Integration = NOT YET PERFORMED"
)

print("=" * 110)

In [ ]:
# ============================================================
# STEP20 - RE-EXPORT PCA VARIANCE
# PNG 600 dpi + PDF
# ============================================================

fig, ax = plt.subplots(
    figsize=(6.4, 5.0)
)

ax.plot(
    pca_variance["PC"],
    pca_variance["CumulativeVariance"],
    marker="o",
    markersize=3.5,
    linewidth=1.4,
    color=NATURE["navy"]
)

ax.set_xlabel(
    "Principal component"
)

ax.set_ylabel(
    "Cumulative explained variance"
)

ax.set_title(
    "PCA cumulative explained variance",
    pad=10
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()

save_publication_figure(
    fig,
    "Step20_02_PCA_CumulativeVariance"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP20 - PUBLICATION-STYLE UNINTEGRATED UMAP
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def plot_umap_publication(
    adata,
    color_col,
    filename,
    palette=None,
    title=None,
    point_size=5
):

    ax = sc.pl.embedding(
        adata,
        basis="umap_unintegrated",
        color=color_col,
        palette=palette,
        size=point_size,
        alpha=0.80,
        frameon=False,
        legend_loc="right margin",
        title=title,
        show=False
    )

    fig = ax.figure

    fig.set_size_inches(
        7.2,
        6.0
    )

    fig.tight_layout()

    save_publication_figure(
        fig,
        filename
    )

    plt.show()
    plt.close(fig)


# ------------------------------------------------------------
# 1. Sample
# ------------------------------------------------------------

sample_levels = (
    adata.obs["sample"]
    .astype(str)
    .unique()
    .tolist()
)

sample_palette = (
    NATURE10 * 3
)[:len(sample_levels)]

plot_umap_publication(
    adata,
    color_col="sample",
    filename="Step20_03_UMAP_Unintegrated_BySample",
    palette=sample_palette,
    title="Unintegrated UMAP by sample",
    point_size=4
)


# ------------------------------------------------------------
# 2. Patient
# ------------------------------------------------------------

patient_levels = (
    adata.obs["patient"]
    .astype(str)
    .unique()
    .tolist()
)

patient_palette = (
    NATURE10
)[:len(patient_levels)]

plot_umap_publication(
    adata,
    color_col="patient",
    filename="Step20_04_UMAP_Unintegrated_ByPatient",
    palette=patient_palette,
    title="Unintegrated UMAP by patient",
    point_size=4
)


# ------------------------------------------------------------
# 3. Site
# ------------------------------------------------------------

site_order = [
    x
    for x in [
        "Normal",
        "Tumor",
        "PVTT",
        "Lymph"
    ]
    if x in adata.obs["site"].astype(str).unique()
]

site_palette = [
    SITE_COLORS[x]
    for x in site_order
]

adata.obs["site"] = pd.Categorical(
    adata.obs["site"],
    categories=site_order
)

plot_umap_publication(
    adata,
    color_col="site",
    filename="Step20_05_UMAP_Unintegrated_BySite",
    palette=site_palette,
    title="Unintegrated UMAP by anatomical site",
    point_size=4
)


# ------------------------------------------------------------
# 4. Original cell type
# ------------------------------------------------------------

celltype_order = [
    x
    for x in [
        "Hepatocyte",
        "T/NK",
        "Myeloid",
        "B",
        "Endothelial",
        "Fibroblast"
    ]
    if x in adata.obs["celltype"].astype(str).unique()
]

celltype_palette = [
    CELLTYPE_COLORS[x]
    for x in celltype_order
]

adata.obs["celltype"] = pd.Categorical(
    adata.obs["celltype"],
    categories=celltype_order
)

plot_umap_publication(
    adata,
    color_col="celltype",
    filename="Step20_06_UMAP_Unintegrated_ByOriginalCellType",
    palette=celltype_palette,
    title="Unintegrated UMAP by original cell type",
    point_size=4
)

print(
    "All Step20 UMAP figures exported "
    "as 600-dpi PNG + vector PDF."
)

In [ ]:
# ============================================================
# STEP20 - LOCK INTEGRATION DECISION
# ============================================================

import json

integration_decision = {

    "Dataset":
        "GSE149614",

    "Step":
        "Step20_Normalization_Integration",

    "Cells":
        int(adata.n_obs),

    "Genes":
        int(adata.n_vars),

    "Normalization":
        "normalize_total(target_sum=10000) + log1p",

    "HVG_method":
        "Seurat dispersion-based HVG",

    "HVG_batch_key":
        "sample",

    "HVG_number":
        int(
            adata.var[
                "highly_variable"
            ].sum()
        ),

    "PCA_components_calculated":
        50,

    "PCs_used_for_neighbors":
        30,

    "Integration_performed":
        False,

    "Primary_representation":
        "Unintegrated PCA",

    "Primary_neighbor_graph":
        "neighbors_unintegrated",

    "Primary_UMAP":
        "X_umap_unintegrated",

    "Global_silhouette": {
        row["Grouping"]:
            float(row["Silhouette"])
        for _, row
        in silhouette_summary.iterrows()
    },

    "Key_observation":
        (
            "Global structure is primarily associated "
            "with cell type rather than sample. "
            "Strong sample-specific separation is mainly "
            "observed within hepatocytes."
        ),

    "Reason_no_global_integration":
        (
            "Sample, patient and anatomical site are "
            "partially confounded. Global correction by "
            "sample could remove genuine patient-, tumor-, "
            "PVTT- or metastasis-associated biology. "
            "Therefore unintegrated representation is retained "
            "for primary major-cell annotation."
        ),

    "Hepatocyte_policy":
        (
            "Reassess batch/patient/site effects after "
            "hepatocyte/malignant-cell extraction. "
            "Any later correction will be performed locally "
            "and evaluated against biological preservation."
        ),

    "Optional_sensitivity_analysis":
        (
            "Integrated representations may be generated later "
            "as sensitivity analyses but will not replace the "
            "primary unintegrated representation without evidence."
        ),

    "Next_step":
        "Step21_Major_Cell_Annotation"
}


INTEGRATION_DECISION_PATH = (
    RESULTS_DIR /
    "Step20_Integration_Decision.json"
)

with open(
    INTEGRATION_DECISION_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        integration_decision,
        f,
        ensure_ascii=False,
        indent=4
    )

print(
    json.dumps(
        integration_decision,
        ensure_ascii=False,
        indent=4
    )
)

In [ ]:
# ============================================================
# STEP20 - SAVE FINAL PRIMARY OBJECT
# ============================================================

FINAL_STEP20_H5AD = (
    RESULTS_DIR /
    "GSE149614_STEP20_UNINTEGRATED_PRIMARY.h5ad"
)

adata.write_h5ad(
    FINAL_STEP20_H5AD,
    compression="gzip"
)

print("=" * 90)
print("STEP20 PRIMARY OBJECT SAVED")
print("=" * 90)

print(FINAL_STEP20_H5AD)

print("\nShape:")
print(adata.shape)

print(
    "\nHVGs:",
    f"{int(adata.var['highly_variable'].sum()):,}"
)

print(
    "PCA:",
    adata.obsm[
        "X_pca_unintegrated"
    ].shape
)

print(
    "UMAP:",
    adata.obsm[
        "X_umap_unintegrated"
    ].shape
)

In [ ]:
# ============================================================
# STEP20 - FINAL SUMMARY
# ============================================================

print("=" * 100)

print("STEP20 STATUS = COMPLETE")

print("\nINPUT:")
print(
    "71,782 clean cells × "
    "25,712 genes"
)

print("\nNORMALIZATION:")
print(
    "Library-size normalization "
    "to 10,000 counts/cell"
)
print("log1p transformation")

print("\nHVG:")
print(
    "3,000 sample-aware HVGs"
)

print("\nPCA:")
print(
    "50 PCs calculated"
)
print(
    "30 PCs used for neighbor graph"
)

print("\nBATCH / INTEGRATION DECISION:")
print(
    "Global integration = NOT performed"
)

print(
    "Primary representation = "
    "unintegrated PCA / UMAP"
)

print("\nRATIONALE:")
print(
    "Overall sample separation is weak, "
    "while cell-type separation is clear."
)

print(
    "Sample-specific separation is mainly "
    "observed in hepatocytes."
)

print(
    "Because sample/patient/site are "
    "partially confounded, global batch "
    "correction risks removing genuine biology."
)

print("\nHEPATOCYTE PLAN:")
print(
    "Reassess correction locally after "
    "malignant/hepatocyte extraction."
)

print("\nNEXT:")
print(
    "Step21 — Major Cell Annotation"
)

print("=" * 100)